# DAPSTOM 6.4 Access Database - Initial EDA

This notebook summarises the Microsoft Access copy received from Cefas. It is intentionally conservative: the source `.accdb` is not overwritten, and the analysis uses derived CSV summaries under `tables/` rather than a full database export.

**Core question:** is the database suitable for constructing predator-prey interaction datasets and later directed food-web/link-prediction experiments?

## Executive Summary

- Core relational chain: `HAUL -> PREDATOR -> PREY`.
- Scale: **10,262 hauls**, **132,647 predator records**, and **283,121 prey records**.
- Coverage spans **1836-2023** in the haul table.
- Spatial resolution is mixed: 66.3% of hauls have point latitude/longitude, 84.8% have ICES rectangle, and all hauls have ICES division/sea.
- There are **17,824** observed predator-prey name pairs, or **17,043** after removing prey records with negative TSN codes.
- `min_num` and `cpw` are almost complete in the prey table, but `pooled` and `num_stomachs` must be handled explicitly because pooled records represent many stomachs.
- The dataset is viable for food-web construction, but empty/unidentified/digested categories should be separated from ordinary trophic edges.

## Reproducibility

The summaries were produced from the working Access copy:

`data/dapstom_6_4_combined_working_copy.accdb`

The extractor uses the DBeaver-downloaded `io.github.spannm:ucanaccess:5.1.5` JDBC driver. To regenerate the CSV summaries from the repo root:

```bash
bash data/processed/dapstom_eda/tools/run_extractor.sh
python3 data/processed/dapstom_eda/tools/build_notebook.py
```

Known non-blocking warning: UCanAccess cannot load the saved Access view `Query1` because the view contains duplicate output column names (`tsn` from predator and prey). This EDA uses base tables and explicit aliases, so the warning is not material.

In [ ]:
from pathlib import Path
import csv, json

TABLE_DIR = Path('tables')
if not TABLE_DIR.exists():
    TABLE_DIR = Path('data/processed/dapstom_eda/tables')

def load_csv(name):
    with (TABLE_DIR / f'{name}.csv').open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

sorted(p.name for p in TABLE_DIR.glob('*.csv'))

## 1. Table Inventory

The Access file exposes ten user tables. The core analysis tables are `HAUL`, `PREDATOR`, `PREY`, `PROVENANCE`, `QUALIFYER`, and `MURRAY_TAXONOMY`.

| Table | Rows |
| --- | --- |
| HAUL 6-4 COMBINED | 10,262 |
| ICES 6-4 COMBINED | 10,261 |
| MURRAY_TAXONOMY | 1,805 |
| PREDATOR 6-4 COMBINED | 132,647 |
| PRED_TAXONOMY 6-4 COMBINED | 210 |
| PREY 6-4 COMBINED | 283,121 |
| PREY_TAXONOMY 6-4 COMBINED | 1,718 |
| PROVENANCE 6-4 COMBINED | 741 |
| QUALIFYER 6-4 | 21 |
| SHIPS 6-4 COMBINED | 93 |

## 2. Spatial and Temporal Coverage

Spatial resolution varies by record. This matters because later networks may need to be built at different spatial grains depending on the coverage required.

| Resolution | Hauls | Coverage |
| --- | --- | --- |
| lat_lon | 6,799 | 66.3% |
| ices_rectangle | 8,698 | 84.8% |
| ices_division | 10,262 | 100.0% |
| sea | 10,262 | 100.0% |

Top sea-level groups by haul count:

| Sea | Hauls | With lat/lon | With ICES rect | Distinct ICES rects |
| --- | --- | --- | --- | --- |
| North Sea | 3,665 | 67.8% | 97.1% | 182 |
| Celtic Sea | 1,513 | 30.9% | 31.3% | 57 |
| Irish Sea | 1,054 | 75.8% | 83.1% | 26 |
| Channel | 1,006 | 30.5% | 98.4% | 24 |
| Spitzbergen (Greenland Sea) | 924 | 97.9% | 97.9% | 94 |
| Freshwater | 426 | 94.8% | 94.8% | 40 |
| W Scotland | 333 | 74.8% | 85.0% | 39 |
| Norwegian Sea | 276 | 94.9% | 99.6% | 171 |
| Faeroes | 196 | 93.9% | 100.0% | 21 |
| Kattegat | 160 | 100.0% | 100.0% | 10 |

## 3. Pooled vs Individual Stomachs

The `PREDATOR` table has both individual and pooled records. For modelling, predator rows should not be interpreted as equal sampling units without considering `pooled` and `num_stomachs`.

| Pooled | Predator rows | % predator rows | Total stomachs | % stomachs | Avg num_stomachs |
| --- | --- | --- | --- | --- | --- |
| n | 122,764 | 92.5% | 122,864 | 25.5% | 1.00 |
| y | 9,879 | 7.4% | 358,838 | 74.5% | 36.32 |
| (missing) | 4 | 0.0% | 229 | 0.0% | 57.25 |

## 4. Prey Evidence and Taxonomy

`PREY.min_num` is present for **283,117/283,121** prey records and sums to **24,831,149**. `PREY.cpw` is present for **283,117/283,121** records.

All prey rows have a TSN and match `MURRAY_TAXONOMY` by TSN in this extract. APHIA IDs are present for **229,247/283,121** prey rows (81.0%).

Most prey records have qualifier `Q1` (`NONE`), but life-stage and sex qualifiers are common enough to preserve during cleaning.

| Qualifier | Description | Prey rows | Total min_num |
| --- | --- | --- | --- |
| Q1 | NONE | 246,642 | 23,903,421 |
| Q6 | ADULT-FEMALE | 9,414 | 9,414 |
| Q9 | COPEPODITE 1-3 | 4,379 | 4,379 |
| Q5 | ADULT | 4,353 | 14,542 |
| Q4 | EGGS | 4,341 | 189,698 |
| Q2 | LARVAE | 4,175 | 399,618 |
| Q10 | COPEPODITE 4-5 | 2,503 | 2,503 |
| Q7 | ADULT-MALE | 2,427 | 2,427 |
| Q8 | COPEPODITE | 910 | 169,155 |
| Q18 | JUVENILE | 880 | 6,062 |

## 5. Missingness in Critical Fields

Core IDs and predator/prey names are complete in the inspected fields. Main gaps are finer time fields, point coordinates, and ICES rectangle blanks.

| Table | Field | Rows | Null rows | Blank text rows |
| --- | --- | --- | --- | --- |
| HAUL 6-4 COMBINED | Month | 10,262 | 510 | 0 |
| HAUL 6-4 COMBINED | Day | 10,262 | 574 | 0 |
| HAUL 6-4 COMBINED | ices_rect | 10,262 | 1 | 1,563 |
| HAUL 6-4 COMBINED | shot_lat_dd | 10,262 | 3,454 | 0 |
| HAUL 6-4 COMBINED | shot_lon_dd | 10,262 | 3,463 | 0 |
| PREDATOR 6-4 COMBINED | pooled | 132,647 | 4 | 0 |
| PREDATOR 6-4 COMBINED | num_empty | 132,647 | 99 | 0 |
| PREY 6-4 COMBINED | min_num | 283,121 | 4 | 0 |
| PREY 6-4 COMBINED | cpw | 283,121 | 4 | 0 |

## 6. Dominant Predators and Prey

Predator codes/names are short labels in the Access table. These should be reconciled with taxonomy before manuscript-quality figures.

Top predator records:

| Predator | Rows | Total stomachs | Avg mean length cm |
| --- | --- | --- | --- |
| COD | 32,610 | 78,956 | 56.4 |
| WHG | 22,671 | 36,173 | 24.4 |
| DAB | 10,080 | 24,504 | 20.8 |
| PLE | 9,628 | 44,180 | 27.8 |
| HAD | 8,694 | 21,225 | 37.4 |
| MAC | 5,933 | 7,535 | 33.2 |
| GUG | 5,929 | 8,892 | 22.1 |
| MEG | 3,870 | 3,926 | 32.1 |
| HER | 2,816 | 77,457 | 15.7 |
| WEL | 2,355 | 2,709 | 12.0 |

Top prey records:

| Prey | Prey TSN | Rows | Total min_num | Total cpw |
| --- | --- | --- | --- | --- |
| Empty | -99901.0 | 33,811 | 109,896 | 0.0 |
| SAN | 171671.0 | 7,937 | 11,162 | 29,902.4 |
| Pseudocalanus elongatus-Adult (Female) | 85370.0 | 7,076 | 7,076 | 0.4 |
| Fish remains | 161105.0 | 5,336 | 8,419 | 237,857.5 |
| EH-Empty | -99901.0 | 4,966 | 4,967 | 0.0 |
| EH-Euphausids | 95573.0 | 4,635 | 14,139 | 1,473.4 |
| Unidentified copepod-Copepodite 1-3 | 85257.0 | 4,373 | 4,373 | 0.3 |
| Copepod | 85258.0 | 3,591 | 50,448 | 80.5 |
| Digested remains | -99904.0 | 3,391 | 9,748 | 6,858.2 |
| Sandeel | 171671.0 | 2,849 | 4,573 | 9,997.0 |

## 7. Non-trophic or Ambiguous Prey Categories

Negative prey TSN values encode categories such as empty stomachs, digested remains, unknown material, or other non-standard entries. These are valuable for sampling/absence information but should usually be filtered or modelled separately from trophic edges.

| Prey TSN | Prey name | Rows | Total min_num |
| --- | --- | --- | --- |
| -99901.0 | Empty | 33,811 | 109,896 |
| -99901.0 | EH-Empty | 4,966 | 4,967 |
| -99904.0 | Digested remains | 3,391 | 9,748 |
| -99904.0 | Unidentifiable (ui) | 1,947 | 1,949 |
| -99901.0 | EH Empty | 1,314 | 1,314 |
| -99904.0 | Unknown | 1,217 | 1,220 |
| -99915.0 | Invertebrate-Egg | 564 | 564 |
| -99907.0 | Sand | 534 | 1,547 |
| -99901.0 | MTY | 501 | 501 |
| -99904.0 | Unidentified rests | 373 | 373 |

## 8. Predator-Prey Edge Potential

The raw joined data imply **17,824** unique predator-prey name pairs. After retaining only positive prey TSN values, the potential edge set is **17,043** pairs. This is promising for link prediction, but edge definitions should be stratified by space/time/source and should not mix pooled and individual evidence without weighting or uncertainty handling.

Most frequent observed predator-prey rows:

| Predator | Prey | Rows | Total min_num | Total cpw |
| --- | --- | --- | --- | --- |
| WHG | Empty | 9,570 | 12,909 | 0.0 |
| COD | EH-Euphausids | 4,379 | 13,852 | 1,443.4 |
| COD | EH-Empty | 4,090 | 4,091 | 0.0 |
| WHG | Pseudocalanus elongatus-Adult (Female) | 3,908 | 3,908 | 0.2 |
| HAD | SAN | 3,582 | 4,759 | 9,464.1 |
| WHG | Unidentified copepod-Copepodite 1-3 | 3,196 | 3,196 | 0.2 |
| COD | Pseudocalanus elongatus-Adult (Female) | 3,168 | 3,168 | 0.2 |
| COD | EH-Fish remains | 2,650 | 2,821 | 230,355.2 |
| GUG | Empty | 2,312 | 3,046 | 0.0 |
| PLE | Empty | 2,234 | 9,040 | 0.0 |
| DAB | Empty | 2,130 | 5,994 | 0.0 |
| COD | Fish remains | 1,982 | 2,790 | 122,335.5 |

## 9. Candidate Strata for Food-web Construction

A practical first pass is to construct networks by `sea x decade`, then later test ICES division or ICES rectangle strata where sample sizes are sufficient. The table below ranks sea-decade strata by unique predator-prey pairs.

| Sea | Decade | Unique pairs | Prey records |
| --- | --- | --- | --- |
| North Sea | 1970 | 2,136 | 18,100 |
| North Sea | 1900 | 2,019 | 8,161 |
| North Sea | 1990 | 1,145 | 32,531 |
| North Sea | 2000 | 1,126 | 53,465 |
| Irish Sea | 1980 | 1,114 | 8,733 |
| Celtic Sea | 1990 | 980 | 11,566 |
| Channel | 1910 | 848 | 3,092 |
| North Sea | 1880 | 833 | 3,826 |
| Irish Sea | 2010 | 681 | 6,718 |
| North Sea | 1980 | 655 | 9,370 |
| Celtic Sea | 2010 | 602 | 3,196 |
| Freshwater | 1960 | 600 | 1,446 |

## 10. Recommendations for the Next Analysis Step

1. Preserve the relational export as base tables rather than creating one flat master CSV.
2. Build a derived edge table with explicit columns: haul context, predator ID/name/TSN, prey name/TSN, `min_num`, `cpw`, `pooled`, `num_stomachs`, `num_empty`, provenance and spatial grain.
3. Treat `Empty`, negative TSN categories, unknown/digested remains, and broad categories separately.
4. For ML link prediction, define positives from observed predator-prey pairs after filtering, and sample negatives within ecologically comparable strata to avoid trivial absences.
5. Use grouped evaluation splits by sea/decade/source/predator where possible; random edge splits will likely overestimate generalisation.
6. Discuss interpretation with Cefas before final modelling choices, especially around pooled records and geographic resolution.

## Generated Files

- `tables/table_inventory.csv` and `tables/column_inventory.csv`: schema overview.
- `tables/table_row_counts.csv`: row counts for all user tables.
- `tables/*summary*.csv`, `tables/top_*.csv`, and `tables/*coverage*.csv`: EDA summaries.
- `tools/DapstomEdaExtractor.java`: JDBC extractor used to create the summaries.
- `tools/run_extractor.sh`: convenience wrapper for recompilation and extraction.